<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/02_Intermediate/L23_Multi_Task_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L23: Multi-Task Learning - One Model, Many Tasks

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Intermediate  
**Lesson:** 23 of 30

---

## Learning Objectives

By the end of this lesson, you will:
- Train a single model on multiple tasks (e.g. sentiment + topic)
- Sample tasks per batch and combine losses
- Understand task balancing and negative transfer

---

## Concept: Multi-Task Learning (MTL)

**MTL** shares a backbone across tasks and uses task-specific heads. Benefits: shared representations, data efficiency. Challenges: task balancing (loss scale, sampling), negative transfer. We combine two classification datasets and train with a shared encoder.

---

In [ ]:
!pip install transformers torch datasets -q
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from datasets import load_dataset
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Two Tasks - Sentiment and (Simulated) Topic

We use sentiment (IMDB) and simulate a second task (e.g. topic) with the same text but different labels to illustrate MTL structure.

---

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

sentiment_ds = load_dataset("imdb", split="train", trust_remote_code=True).select(range(400))
sentiment_ds = sentiment_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
sentiment_ds = sentiment_ds.rename_column("label", "sentiment")
sentiment_ds = sentiment_ds.add_column("topic", [i % 3 for i in range(len(sentiment_ds))])
sentiment_ds.set_format("torch")
print("Combined dataset: sentiment (0/1) + topic (0/1/2). Size:", len(sentiment_ds))

## Part 2: Shared Encoder + Two Heads

Backbone (e.g. DistilBERT) outputs pooled representation; two linear heads for sentiment (2 classes) and topic (3 classes).

---

In [ ]:
class MTLModel(nn.Module):
    def __init__(self, num_sentiment=2, num_topic=3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("distilbert-base-uncased")
        hidden = self.backbone.config.hidden_size
        self.sentiment_head = nn.Linear(hidden, num_sentiment)
        self.topic_head = nn.Linear(hidden, num_topic)

    def forward(self, input_ids, attention_mask, sentiment=None, topic=None):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0]
        loss = None
        loss_s = nn.functional.cross_entropy(self.sentiment_head(pooled), sentiment) if sentiment is not None else None
        loss_t = nn.functional.cross_entropy(self.topic_head(pooled), topic) if topic is not None else None
        if loss_s is not None and loss_t is not None:
            loss = loss_s + loss_t
        return {"loss": loss, "sentiment_logits": self.sentiment_head(pooled), "topic_logits": self.topic_head(pooled)}

model = MTLModel()
print("MTL model: shared backbone + sentiment head + topic head.")

## Part 3: Training Loop with Combined Loss

---

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
loader = DataLoader(sentiment_ds, batch_size=16, shuffle=True)

model.train()
for epoch in range(2):
    total = 0.0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        loss = out["loss"]
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total += loss.item()
    print(f"Epoch {epoch+1} avg loss: {total/len(loader):.4f}")
print("MTL training done. Tune loss weights (e.g. 0.5*sentiment + 0.5*topic) if one task dominates.")

## Exercises

1. Add a third task (e.g. NER or another classification) and a third head.
2. Sample tasks with different probabilities (e.g. 70% sentiment, 30% topic) and compare.
3. Use uncertainty weighting or GradNorm to balance task losses automatically.

---

## Key Takeaways

1. MTL = one backbone + multiple heads; loss = sum (or weighted sum) of task losses.
2. Balance tasks by loss scale or sampling to avoid one task dominating.
3. Negative transfer: if tasks conflict, consider separate models or smaller shared layers.

---

## Next Lesson

**L24: Instruction Tuning** – Fine-tuning models to follow instructions.

---